# Agent and Tools Notebook

The first notebook already handled the CUAD data pipeline:
- loaded the CUAD dataset
- cleaned the text
- chunked the contracts
- created embeddings
- built the FAISS index
- tested retrieval

In this notebook I am using those saved files to build the agent layer:
- scope check
- contract retrieval tool
- risk assessment tool
- GPT-4o and GPT-4o-mini agent calls
- contract question examples
- irrelevant question rejection examples
- saved outputs for the evaluation notebook

In [0]:
%pip install -q openai faiss-cpu sentence-transformers pandas numpy

In [0]:
import os
import pickle
import json
import time

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [0]:
chunks_file = "cuad_chunks.pkl"
faiss_file = "cuad_faiss_index.index"

def find_file(filename):
    # First try the current folder.
    if os.path.exists(filename):
        return filename 

    raise FileNotFoundError(f"Could not find {filename}. Run the data pipeline notebook first.")

chunks_path = find_file(chunks_file)
faiss_path = find_file(faiss_file)

with open(chunks_path, "rb") as f:
    chunks = pickle.load(f)

index = faiss.read_index(faiss_path)

print("Chunks loaded:", len(chunks))
print("FAISS vectors loaded:", index.ntotal)
print("Sample chunk:")
print(chunks[0][:500])

In [0]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded.")

In [0]:
def search_contracts(query, k=5):
    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k=k
    )

    results = []

    for idx, distance in zip(indices[0], distances[0]):
        results.append({
            "distance": float(distance),
            "text": chunks[idx]
        })

    return results

# test
results = search_contracts("termination clause", k=3)

for r in results:
    print("\nDistance:", r["distance"])
    print(r["text"][:500])

In [0]:
def check_scope(question):
    question_lower = question.lower()

    contract_terms = [
        "contract", "agreement", "clause", "provision", "termination",
        "indemnification", "liability", "confidentiality", "payment",
        "obligation", "breach", "damages", "warranty", "assignment",
        "renewal", "risk", "legal", "compliance"
    ]

    unrelated_terms = [
        "trip", "travel", "vacation", "restaurant", "recipe", "diagnose",
        "medical", "chest pain", "weather", "sports", "movie", "shopping"
    ]

    if any(term in question_lower for term in unrelated_terms):
        return False

    if any(term in question_lower for term in contract_terms):
        return True

    return False

In [0]:
def retrieve_clauses(question, k=5):
    return search_contracts(question, k=k)

In [0]:
def assess_risk(question, retrieved_clauses):
    combined_text = " ".join([r["text"] for r in retrieved_clauses]).lower()
    question_lower = question.lower()

    high_risk_terms = [
        "indemnification", "indemnify", "unlimited liability",
        "liquidated damages", "breach", "penalty", "damages",
        "without notice", "sole discretion"
    ]

    medium_risk_terms = [
        "termination", "liability", "confidentiality", "assignment",
        "governing law", "warranty", "renewal", "payment", "notice"
    ]

    high_matches = []
    medium_matches = []

    for term in high_risk_terms:
        if term in combined_text or term in question_lower:
            high_matches.append(term)

    for term in medium_risk_terms:
        if term in combined_text or term in question_lower:
            medium_matches.append(term)

    if len(high_matches) >= 2:
        risk_level = "High"
    elif len(high_matches) == 1 or len(medium_matches) >= 2:
        risk_level = "Medium"
    elif len(medium_matches) == 1:
        risk_level = "Low to Medium"
    else:
        risk_level = "Low"

    return {
        "risk_level": risk_level,
        "high_matches": high_matches,
        "medium_matches": medium_matches
    }

In [0]:
def prepare_agent_context(question, k=5):
    start_time = time.time()

    if not check_scope(question):
        return {
            "status": "rejected",
            "question": question,
            "answer": (
                "I can only help with contract analysis questions. "
                "Please ask about contract clauses, obligations, risks, or agreement terms."
            ),
            "latency_seconds": round(time.time() - start_time, 3)
        }

    retrieved_clauses = retrieve_clauses(question, k=k)
    risk_result = assess_risk(question, retrieved_clauses)

    return {
        "status": "ready_for_llm",
        "question": question,
        "retrieved_clauses": retrieved_clauses,
        "risk_result": risk_result,
        "latency_seconds": round(time.time() - start_time, 3)
    }

In [0]:
contract_questions = [
    "Does this contract contain risky termination clauses?",
    "What indemnification obligations are present?",
    "Summarize the key risks in this agreement."
]

prep_results = []

for question in contract_questions:
    result = prepare_agent_context(question, k=5)

    prep_results.append({
        "question": question,
        "status": result["status"],
        "risk_level": result.get("risk_result", {}).get("risk_level"),
        "latency_seconds": result["latency_seconds"],
        "top_retrieved_clause": result.get("retrieved_clauses", [{}])[0].get("text", "")[:1000]
    })

prep_results_df = pd.DataFrame(prep_results)
prep_results_df

In [0]:
for result in prep_results:
    print("=" * 100)
    print("Question:", result["question"])
    print("Status:", result["status"])
    print("Risk level:", result["risk_level"])
    print("Latency:", result["latency_seconds"])
    print("\nTop retrieved clause:")
    print(result["top_retrieved_clause"])
    print()

In [0]:
irrelevant_questions = [
    "Can you help me plan a trip to Italy?",
    "Can you diagnose my chest pain?"
]

rejection_results = []

for question in irrelevant_questions:
    result = prepare_agent_context(question, k=5)

    rejection_results.append({
        "question": question,
        "status": result["status"],
        "answer": result["answer"],
        "latency_seconds": result["latency_seconds"]
    })

rejection_df = pd.DataFrame(rejection_results)
rejection_df

In [0]:
output_dir = "agent_outputs"
os.makedirs(output_dir, exist_ok=True)

prep_results_df.to_csv(os.path.join(output_dir, "agent_context_prep_results.csv"), index=False)
rejection_df.to_csv(os.path.join(output_dir, "rejection_examples.csv"), index=False)

with open(os.path.join(output_dir, "agent_context_prep_results.json"), "w") as f:
    json.dump(prep_results, f, indent=2)

with open(os.path.join(output_dir, "rejection_examples.json"), "w") as f:
    json.dump(rejection_results, f, indent=2)

print("Saved API-free handoff outputs to:", output_dir)